In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import statistics
import matplotlib.pyplot as plt

from src.population_forecaster import (
    DistrictPopulation,
    LinearTrendForecaster,
    CAGRForecaster,
    FibonacciForecaster,
    bootstrap_prediction_interval
)

print("Project 1 setup successful!")

In [ ]:
years = np.arange(2015, 2025)

kampala = np.array([
    1200, 1250, 1300, 1350, 1420,
    1500, 1580, 1650, 1720, 1800
])

wakiso = np.array([
    950, 1000, 1070, 1150, 1220,
    1300, 1390, 1480, 1570, 1670
])

gulu = np.array([
    320, 330, 345, 360, 375,
    390, 410, 430, 455, 480
])

mbarara = np.array([
    450, 470, 495, 520, 550,
    580, 610, 640, 675, 710
])

jinja = np.array([
    380, 400, 425, 450, 475,
    505, 535, 565, 600, 635
])

print(years)
print("Data loaded successfully!")

In [ ]:
districts = {
    "Kampala": DistrictPopulation("Kampala", years, kampala),
    "Wakiso": DistrictPopulation("Wakiso", years, wakiso),
    "Gulu": DistrictPopulation("Gulu", years, gulu),
    "Mbarara": DistrictPopulation("Mbarara", years, mbarara),
    "Jinja": DistrictPopulation("Jinja", years, jinja)
}

print(districts)

In [ ]:
print(districts["Kampala"])
print("Number of observations:", len(districts["Kampala"]))

In [ ]:
#--- Calculating the descriptive statistics ---

for name, district in districts.items():
    data = district.populations

    print(f"\n{name}")
    print("Mean:", statistics.mean(data))
    print("Median:", statistics.median(data))
    print("Variance:", statistics.variance(data))
    print("Standard deviation:", statistics.stdev(data))

In [ ]:
#--- Comparing NumPy with statistics---

for name, district in districts.items():
    data = district.populations

    print(f"\n{name}")
    print("statistics variance:", statistics.variance(data))
    print("NumPy variance:", np.var(data))
    print("NumPy sample variance:", np.var(data, ddof=1))
    
    print("statistics stdev:", statistics.stdev(data))
    print("NumPy std:", np.std(data))
    print("NumPy sample std:", np.std(data, ddof=1))

In [ ]:
#--- Calculate year-on-year growth ---

for name, district in districts.items():
    growth = district.growth_rates() * 100

    print(f"\n{name}")
    print("Year-on-year growth rates (%):")
    print(np.round(growth, 2))

In [ ]:
#--- calculate the compound annual growth rate from 2015 to 2024 for each district ---

for name, district in districts.items():
    cagr = district.cagr() * 100

    print(f"{name}: CAGR = {cagr:.2f}%")


In [ ]:
#--- Calculate the average year-on-year growth ---

for name, district in districts.items():
    average_growth = np.mean(district.growth_rates()) * 100

    print(f"{name}: Average YoY growth = {average_growth:.2f}%")


In [ ]:
#--- Checking the abstract forecaster classes  ---

linear = LinearTrendForecaster()
cagr_model = CAGRForecaster()
fib_model = FibonacciForecaster()

print("Forecaster classes created successfully!")

In [ ]:
#--- Prepare the training and test data ---

train_mask = years <= 2021
test_mask = years >= 2022

train_years = years[train_mask]
test_years = years[test_mask]

print("Training years:", train_years)
print("Testing years:", test_years)

In [ ]:
#--- Testing models for Kampala district---

kampala_train = kampala[train_mask]

linear = LinearTrendForecaster().fit(train_years, kampala_train)
cagr_model = CAGRForecaster().fit(train_years, kampala_train)
fib_model = FibonacciForecaster().fit(train_years, kampala_train)

print("Models fitted successfully!")

In [ ]:
#--- Generating Kampala test predictions ---

linear_pred = linear.predict(3)
cagr_pred = cagr_model.predict(3)
fib_pred = fib_model.predict(3)

print("Actual:", kampala[test_mask])
print("Linear:", np.round(linear_pred, 2))
print("CAGR:", np.round(cagr_pred, 2))
print("Fibonacci:", np.round(fib_pred, 2))

In [ ]:
#--- Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Mean Absolute Percentage Error () for Kampala  ---

actual = kampala[test_mask]

for model_name, predictions in {
    "Linear": linear_pred,
    "CAGR": cagr_pred,
    "Fibonacci": fib_pred
}.items():

    mae = np.mean(np.abs(actual - predictions))
    rmse = np.sqrt(np.mean((actual - predictions) ** 2))
    mape = np.mean(
        np.abs((actual - predictions) / actual)
    ) * 100

    print(f"\n{model_name}")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAPE: {mape:.2f}%")

In [ ]:
#--- Generatingtest predictions for Kampala, Wakiso, Gulu, Mbarara and Jinja (All 5 districts) ---

for name, district in districts.items():
    train_data = district.populations[train_mask]
    actual = district.populations[test_mask]

    linear_model = LinearTrendForecaster().fit(train_years, train_data)
    cagr_model = CAGRForecaster().fit(train_years, train_data)
    fib_model = FibonacciForecaster().fit(train_years, train_data)

    predictions = {
        "Linear": linear_model.predict(3),
        "CAGR": cagr_model.predict(3),
        "Fibonacci": fib_model.predict(3)
    }

    print(f"\n{name}")

    for model_name, pred in predictions.items():
        mae = np.mean(np.abs(actual - pred))
        rmse = np.sqrt(np.mean((actual - pred) ** 2))
        mape = np.mean(np.abs((actual - pred) / actual)) * 100

        print(
            f"{model_name}: "
            f"MAE={mae:.2f}, "
            f"RMSE={rmse:.2f}, "
            f"MAPE={mape:.2f}%"
        )

In [ ]:
# --- My results table ---

results = [
    ["Kampala", "Linear", 37.62, 38.97, 2.17],
    ["Kampala", "CAGR", 9.62, 10.39, 0.55],
    ["Kampala", "Fibonacci", 3543.33, 4025.36, 202.00],

    ["Wakiso", "Linear", 49.40, 52.37, 3.09],
    ["Wakiso", "CAGR", 6.80, 8.05, 0.42],
    ["Wakiso", "Fibonacci", 3060.00, 3479.87, 189.87],

    ["Gulu", "Linear", 18.57, 20.29, 4.01],
    ["Gulu", "CAGR", 9.44, 10.87, 2.02],
    ["Gulu", "Fibonacci", 911.67, 1035.64, 196.04],

    ["Mbarara", "Linear", 15.18, 16.54, 2.21],
    ["Mbarara", "CAGR", 0.68, 1.01, 0.10],
    ["Mbarara", "Fibonacci", 1358.33, 1543.38, 197.10],

    ["Jinja", "Linear", 17.68, 19.18, 2.89],
    ["Jinja", "CAGR", 0.66, 0.84, 0.11],
    ["Jinja", "Fibonacci", 1183.33, 1344.95, 192.71]
]

results_df = pd.DataFrame(
    results,
    columns=["District", "Model", "MAE", "RMSE", "MAPE (%)"]
)

results_df



In [ ]:
#--- Identifying the best model (automatically)

best_models = {}

for district in results_df["District"].unique():
    district_results = results_df[results_df["District"] == district]
    best = district_results.loc[district_results["MAPE (%)"].idxmin()]
    best_models[district] = best["Model"]

print("Best model for each district:")

for district, model in best_models.items():
    print(f"{district}: {model}")

In [ ]:
#--- Forecast 2025–2029 ---

forecast_years = np.arange(2025, 2030)

forecasts = {}

for name, district in districts.items():
    model = CAGRForecaster().fit(
        district.years,
        district.populations
    )

    forecast = model.predict(5)
    forecasts[name] = forecast

    print(f"\n{name}")
    for year, value in zip(forecast_years, forecast):
        print(f"{year}: {value:.2f} thousand")

In [ ]:
#--- Calculating the 2029 classroom requirement ---
# 18% of the population is primary-school age.
# 53 pupils per classroom.
# We need the estimated additional classrooms by 2029.

for name, forecast in forecasts.items():
    population_2029 = forecast[-1]

    # Convert population from thousands to actual people
    population_people = population_2029 * 1000

    primary_school_age = population_people * 0.18
    classrooms = np.ceil(primary_school_age / 53)

    print(f"{name}:")
    print(f"2029 population: {population_people:,.0f} people")
    print(f"Primary-school-age population: {primary_school_age:,.0f}")
    print(f"Estimated classrooms required: {int(classrooms):,}")
    print()

In [ ]:
#--- Compare the variance of the actual population with the forecast population ---

for name, district in districts.items():
    actual_variance = np.var(district.populations)

    forecast_variance = np.var(forecasts[name])

    print(f"{name}:")
    print(f"Actual population variance: {actual_variance:.2f}")
    print(f"Forecast population variance: {forecast_variance:.2f}")
    print()

In [ ]:
#-- Train/test variables --

train_mask = years <= 2021
test_mask = years >= 2022

train_years = years[train_mask]
test_years = years[test_mask]

print("Training years:", train_years)
print("Testing years:", test_years)

In [ ]:
#-- Testing the bootstrap prediction interval for Kampala district ---

test_actual = np.array([1650, 1720, 1800])
test_forecast = np.array([1654.13, 1731.74, 1812.99])

lower, upper = bootstrap_prediction_interval(
    test_actual,
    test_forecast,
    n_bootstrap=1000,
    seed=42
)

print("Lower 95% bounds:", np.round(lower, 2))
print("Upper 95% bounds:", np.round(upper, 2))


My Bootstrap Prediction Interval Interpretation

The bootstrap procedure generated 95% prediction intervals around the 2022–2024 CAGR forecasts for Kampala. The intervals show the uncertainty associated with the forecasts based on resampled historical residuals. The results demonstrate that the predicted values are not single certain outcomes but have a range of plausible values. However, the intervals are based on only the available historical residuals and therefore may not capture major future demographic or economic changes.

In [ ]:
#--- Figure with subplots Actual population, Fitted values, Forecast values, Train/test split ---

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, district) in enumerate(districts.items()):
    ax = axes[i]

    train_data = district.populations[train_mask]

    # Fit CAGR model on training data
    model = CAGRForecaster().fit(train_years, train_data)

    # Fitted values for training period
    fitted = model.predict(len(train_years) - 1)

    # Forecast 2025-2029
    forecast = forecasts[name]

    # Plot actual population
    ax.plot(
        district.years,
        district.populations,
        marker="o",
        label="Actual"
    )

    # Plot fitted values
    ax.plot(
        train_years[1:],
        fitted,
        linestyle="--",
        label="Fitted"
    )

    # Plot forecast
    ax.plot(
        forecast_years,
        forecast,
        marker="x",
        linestyle=":",
        label="Forecast"
    )

    # Mark train/test boundary
    ax.axvline(
        x=2021.5,
        linestyle="--",
        label="Train/Test Split"
    )

    ax.set_title(name)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population (thousands)")
    ax.legend()
    ax.grid(True)

# Remove the unused sixth subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

### My Forecast Interpretation

The CAGR model was selected for all five districts because it produced the lowest Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Mean Absolute Percentage Error (MAPE) during the 2022–2024 test period. The forecasts indicate continued population growth from 2025 to 2029 across Kampala, Wakiso, Gulu, Mbarara, and Jinja. The forecast variance is lower than the historical variance for every district, suggesting that the CAGR model produces a relatively smooth growth trajectory. However, the forecasts should be interpreted cautiously because they assume that historical compound growth rates will continue into the future and do not account for unexpected demographic, economic, migration, or policy changes.
